In [8]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 1. 设置路径与定义稳健读取函数
# ==========================================
base_path = r'C:\Users\良\Desktop\文章\数据\2018-2020'

def safe_load_xpt(file_name, cols):
    full_path = os.path.join(base_path, file_name)
    if not os.path.exists(full_path):
        print(f"⚠️ 跳过：文件 {file_name} 在本地目录中不存在。")
        return None
    try:
        df = pd.read_sas(full_path)
        # 确保列名是大写或匹配，NHANES 有时会有大小写差异
        available_cols = [c for c in cols if c in df.columns]
        if 'SEQN' not in available_cols:
            available_cols.append('SEQN')
        return df[available_cols]
    except Exception as e:
        print(f"❌ 读取 {file_name} 出错: {e}")
        return None

# ==========================================

In [9]:
# 2. 按照您的截图读取现有文件
# ==========================================
print("开始提取变量...")

# A. 人口统计 (Age, Gender)
demo_df = safe_load_xpt('P_DEMO.xpt', ['SEQN', 'RIDAGEYR', 'RIAGENDR'])

# B. 结局变量: 糖尿病 (DIQ010)
target_df = safe_load_xpt('P_DIQ.xpt', ['SEQN', 'DIQ010'])

# C. 二阶段核心1: SII 组件 (血小板, 中性粒, 淋巴)
cbc_df = safe_load_xpt('P_CBC.xpt', ['SEQN', 'LBXPLTSI', 'LBDNENO', 'LBDLYMNO'])

# D. 二阶段核心2: TyG 组件 (甘油三酯-注意是LBXTR, 空腹血糖)
trig_df = safe_load_xpt('P_TRIGLY.xpt', ['SEQN', 'LBXTR'])
glu_df = safe_load_xpt('P_GLU.xpt', ['SEQN', 'LBXGLU'])

# E. 协变量: BMI, 吸烟, 高血压
bmi_df = safe_load_xpt('P_BMX.xpt', ['SEQN', 'BMXBMI'])
smoke_df = safe_load_xpt('P_SMQ.xpt', ['SEQN', 'SMQ020'])
bp_df = safe_load_xpt('P_BPQ.xpt', ['SEQN', 'BPQ020'])

# F. 可能会缺失的体力活动 (如果文件夹里真没有，程序不会崩)
paq_df = safe_load_xpt('P_PAQ.xpt', ['SEQN', 'PAQ650'])

# ==========================================

开始提取变量...
⚠️ 跳过：文件 P_PAQ.xpt 在本地目录中不存在。


In [10]:
# 3. 仿照您的成功逻辑进行合并
# ==========================================
df = demo_df
# 将所有读取到的 df 放入列表
to_merge = [target_df, cbc_df, trig_df, glu_df, bmi_df, smoke_df, bp_df, paq_df]

for temp_df in to_merge:
    if temp_df is not None:
        temp_df['SEQN'] = temp_df['SEQN'].astype(int)
        df = pd.merge(df, temp_df, on='SEQN', how='left')

# ==========================================
# 4. 数据清洗与计算 (严格确保无缺失)
# ==========================================
# 1. 计算 SII
df['SII'] = (df['LBXPLTSI'] * df['LBDNENO']) / df['LBDLYMNO']

# 2. 计算 TyG (根据您之前的 KeyError 修正)
# TyG = ln[TG(mg/dL) * Glucose(mg/dL) / 2]
df['TyG'] = np.log(df['LBXTR'] * df['LBXGLU'] / 2)

# 3. 结局变量映射
df['target'] = df['DIQ010'].map({1: 1, 2: 0})

# 4. 筛选成年人并剔除缺失值
df = df[df['RIDAGEYR'] > 20].copy()
final_df = df.dropna(subset=['target', 'SII', 'TyG', 'BMXBMI', 'RIDAGEYR'])

# 保存最终合并结果
output_path = os.path.join(base_path, 'nhanes_final_merged.csv')
final_df.to_csv(output_path, index=False)

print("-" * 30)
print(f"合并成功！")
print(f"最终生成的 CSV 包含变量: {list(final_df.columns)}")
print(f"可用完整样本量: {len(final_df)}")

------------------------------
合并成功！
最终生成的 CSV 包含变量: ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'DIQ010', 'LBXPLTSI', 'LBDNENO', 'LBDLYMNO', 'LBXTR', 'LBXGLU', 'BMXBMI', 'SMQ020', 'BPQ020', 'SII', 'TyG', 'target']
可用完整样本量: 3691
